# Operationalizations for Analyses: Dynamic OLS Estimations
---
**Context:** We conduct five comprehensive analyses to operationalize the algorithmic response at different levels. This notebook utilizes Dynamic Ordinary Least Squares (DOLS) regressions to estimate these structural relationships, incorporating leads and lags of differenced regressors to address potential endogeneity.

To preclude the risk of spurious regression in our time-series data, all models are subjected to the **Shin (1994) Test for Residual Stationarity**. The null hypothesis posits that the residuals are stationary; a failure to reject the null ($p > 0.05$) validates the structural integrity of the DOLS parameters.

**Analyses Covered:**
* **(a)** New AIGC entry into exposure tiers (Initial visibility allocation).
* **(b & c)** Relative algorithmic exposure response to relative supply scale and relative consumer preference.
* **(d)** Supply scale elasticity on creator’s overall engagement return.
* **(e)** Supply scale elasticity on consumer’s overall engagement depth.

### Step 1: Global Data Preparation & Temporal Controls
We ingest the daily panel dataset and compute structural exogenous controls (e.g., statutory holidays, day-of-week effects) to account for temporal heterogeneity in online consumption.

In [ ]:
import sys
import pickle
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

sys.path.append('../core')
from regression import enhanced_dols_analysis

# 1. Load Data
df = pd.read_csv('daily_panel.csv')
df['date'] = pd.to_datetime(df['date'])

# 2. Define Statutory Holidays
holiday_dates = [
    '2024-06-08', '2024-06-09', '2024-06-10', '2024-09-15', '2024-09-16', '2024-09-17',
    '2024-10-01', '2024-10-02', '2024-10-03', '2024-10-04', '2024-10-05', '2024-10-06',
    '2024-10-07', '2025-01-01', '2025-01-28', '2025-01-29', '2025-01-30', '2025-01-31',
    '2025-02-01', '2025-02-02', '2025-02-03', '2025-04-04', '2025-04-05', '2025-04-06',
    '2025-05-01', '2025-05-02', '2025-05-03', '2025-05-04', '2025-05-05'
]
holiday_dates = [pd.to_datetime(d) for d in holiday_dates]

df['holiday'] = df['date'].isin(holiday_dates).astype(int)
df['holiday_entry'] = ((df['holiday'] == 1) & (df['holiday'].shift(1) == 0)).astype(int)
df['holiday_exit'] = ((df['holiday'] == 0) & (df['holiday'].shift(1) == 1)).astype(int)

df['wd'] = df['date'].dt.weekday
wd_dummies = pd.get_dummies(df['wd'], prefix='wd', drop_first=True).astype(int)

# Base exogenous controls (used across all specifications)
exog_base = pd.concat([wd_dummies, df['holiday'], df['holiday_entry'], df['holiday_exit']], axis=1)
print("✅ Global temporal controls successfully instantiated.")

### Operationalization (a): New AIGC entry into exposure tiers
Links AIGC supply scale to the entry of new AIGC items into different exposure tiers, representing the initial visibility allocation. The specification is:

$$ \pi^k_t = \alpha_k + \beta_k S^{AIGC}_t + \theta_k S^{HGC}_t + \sum_{j=-p}^{p} \gamma^k_j \Delta S^{AIGC}_{t-j} + \epsilon^k_t + \delta_k D_t $$

where $\pi^k_t$ is the proportion of new AIGC videos entering exposure tier $k$, and $S^{AIGC}_t$ denotes the absolute supply scale of AIGC. The absolute supply scale of HGC, denoted by $S^{HGC}_t$, is considered as the controlled variable.

In [ ]:
print("=" * 80)
print("Operationalization (a): Exposure Tiers")
print("=" * 80)

supply_scales = ['scale_aigc', 'scale_hgc']
# Absolute scales (no log)
X_a = sm.add_constant(pd.concat([df[supply_scales], exog_base], axis=1))

exposure_tiers = {
    'Low (<10 views)': ('exp31_lt10rate_aigc', 'dols_exposure_tier_low.pkl'),
    'Med-Low (11-100 views)': ('exp31_ge10rate_aigc', 'dols_exposure_tier_med_low.pkl'),
    'Med-High (101-1K views)': ('exp31_ge100rate_aigc', 'dols_exposure_tier_med_high.pkl'),
    'High (>1K views)': ('exp31_ge1000rate_aigc', 'dols_exposure_tier_high.pkl')
}

for tier_name, (target_var, out_file) in exposure_tiers.items():
    print(f"\nEstimating Tier: {tier_name}")
    y = df[target_var]
    
    final_model, shin_res, boot_res, partial_res = enhanced_dols_analysis(
        y, X_a, max_lags=5, supply_scales=supply_scales, 
        do_bootstrap=True, bootstrap_B=2000, random_state=42
    )
    
    results_dict = {
        'final_model': final_model, 'shin_results': shin_res,
        'bootstrap_results': boot_res, 'partial_residuals': partial_res,
        'y': y.values, 'X_base_columns': X_a.columns.tolist(),
        'custom_endog_vars': supply_scales, 'date': df['date'].values,
        'metadata': {'description': f'Analysis (a) - {tier_name}'}
    }
    with open(out_file, 'wb') as f:
        pickle.dump(results_dict, f)
    print(f"✅ Shin Test p-value: {shin_res['p_value']:.4f} -> Saved to {out_file}")

### Operationalization (b & c): Algorithmic Exposure Response to Scale-over-Preference Dynamics
Examines the associations of relative supply scale ($S_t$) and relative consumer preference ($P_t$) with the relative median exposure of AIGC ($E_t$):

$$ E_t = \alpha + \beta_S S_t + \beta_P P_t + \sum_{j=-p}^{p} \gamma_{j,S} \Delta S_{t-j} + \sum_{j=-p}^{p} \gamma_{j,P} \Delta P_{t-j} + \epsilon_t + \delta D_t $$

Here, as $S_t$ and $P_t$ jointly determine the Scale-over-Preference (SoP) condition, we analyze them together.

In [ ]:
print("\n" + "=" * 80)
print("Operationalization (b & c): Relative Exposure Dynamics")
print("=" * 80)

relative_vars = ['pref_ratio', 'scale_ratio']
X_bc = sm.add_constant(pd.concat([df[relative_vars], exog_base], axis=1))
y_bc = df['exp31_p50_ratio']

final_model, shin_res, boot_res, partial_res = enhanced_dols_analysis(
    y_bc, X_bc, max_lags=5, supply_scales=relative_vars, 
    do_bootstrap=True, bootstrap_B=2000, random_state=42
)

out_file = 'dols_relative_exposure_dynamics.pkl'
results_dict = {
    'final_model': final_model, 'shin_results': shin_res,
    'bootstrap_results': boot_res, 'partial_residuals': partial_res,
    'y': y_bc.values, 'X_base_columns': X_bc.columns.tolist(),
    'custom_endog_vars': relative_vars, 'date': df['date'].values,
    'metadata': {'description': 'Analysis (b & c) - Relative Exposure'}
}
with open(out_file, 'wb') as f:
    pickle.dump(results_dict, f)
print(f"✅ Shin Test p-value: {shin_res['p_value']:.4f} -> Saved to {out_file}")

### Operationalization (d): Elasticities of Supply Scale on Creator Outcomes
Relates the absolute supply scale to downstream engagement returns for creators via log-log specification:

$$ \ln(ER_t) = \alpha + \beta \ln(S^{AIGC}_t) + \theta \ln(S^{HGC}_t) + \sum_{j=-p}^{q} \gamma_j \Delta \ln(S^{AIGC}_{t-j}) + \epsilon_t + \delta D_t $$

where $ER_t$ represents the overall engagement return for creators on day $t$. For reference, we perform symmetrical analyses for HGC.

In [ ]:
print("\n" + "=" * 80)
print("Operationalization (d): Creator Returns (Elasticities)")
print("=" * 80)

creator_configs = [
    {'desc': 'AIGC Creator Completions', 'target': 'com31_sum_aigc', 'vars': ['scale_aigc', 'scale_hgc'], 'out': 'dols_creator_completions_aigc.pkl'},
    {'desc': 'AIGC Creator Valid-Views', 'target': 'val31_sum_aigc', 'vars': ['scale_aigc', 'scale_hgc'], 'out': 'dols_creator_validviews_aigc.pkl'},
    {'desc': 'HGC Creator Completions', 'target': 'com31_sum_hgc', 'vars': ['scale_hgc', 'scale_aigc'], 'out': 'dols_creator_completions_hgc.pkl'},
    {'desc': 'HGC Creator Valid-Views', 'target': 'val31_sum_hgc', 'vars': ['scale_hgc', 'scale_aigc'], 'out': 'dols_creator_validviews_hgc.pkl'}
]

for config in creator_configs:
    print(f"\nEstimating: {config['desc']}")
    
    # Log transformation per specification (d)
    X_d = sm.add_constant(pd.concat([np.log(df[config['vars']]), exog_base], axis=1))
    y_d = np.log(df[config['target']])
    
    final_model, shin_res, boot_res, partial_res = enhanced_dols_analysis(
        y_d, X_d, max_lags=5, supply_scales=config['vars'], 
        do_bootstrap=True, bootstrap_B=2000, random_state=42
    )
    
    results_dict = {
        'final_model': final_model, 'shin_results': shin_res,
        'bootstrap_results': boot_res, 'partial_residuals': partial_res,
        'y': y_d.values, 'X_base_columns': X_d.columns.tolist(),
        'custom_endog_vars': config['vars'], 'date': df['date'].values,
        'metadata': {'description': f"Analysis (d) - {config['desc']}"}
    }
    with open(config['out'], 'wb') as f:
        pickle.dump(results_dict, f)
    print(f"✅ Shin Test p-value: {shin_res['p_value']:.4f} -> Saved to {config['out']}")

### Operationalization (e): Elasticities of Supply Scale on Consumer Outcomes
Relates the absolute supply scales to consumer-side engagement depth via log-log specification:

$$ \ln(ED_t) = \alpha + \beta_{AIGC} \ln(S^{AIGC}_t) + \beta_{HGC} \ln(S^{HGC}_t) + \dots + \epsilon_t + \delta D_t $$

where $ED_t$ represents the overall consumer engagement depth at day $t$. Since consumer depth is jointly determined by AIGC and HGC, we involve both for the regression.

In [ ]:
print("\n" + "=" * 80)
print("Operationalization (e): Consumer Depth (Elasticities)")
print("=" * 80)

consumer_configs = [
    {'desc': 'Consumer Valid-View Rate', 'target': 'val31_rate_all', 'vars': ['scale_aigc', 'scale_hgc'], 'out': 'dols_consumer_validview_rate.pkl'},
    {'desc': 'Consumer Completion Rate', 'target': 'com31_rate_all', 'vars': ['scale_aigc', 'scale_hgc'], 'out': 'dols_consumer_completion_rate.pkl'}
]

for config in consumer_configs:
    print(f"\nEstimating: {config['desc']}")
    
    # Log transformation per specification (e)
    X_e = sm.add_constant(pd.concat([np.log(df[config['vars']]), exog_base], axis=1))
    y_e = np.log(df[config['target']])
    
    final_model, shin_res, boot_res, partial_res = enhanced_dols_analysis(
        y_e, X_e, max_lags=5, supply_scales=config['vars'], 
        do_bootstrap=True, bootstrap_B=2000, random_state=42
    )
    
    results_dict = {
        'final_model': final_model, 'shin_results': shin_res,
        'bootstrap_results': boot_res, 'partial_residuals': partial_res,
        'y': y_e.values, 'X_base_columns': X_e.columns.tolist(),
        'custom_endog_vars': config['vars'], 'date': df['date'].values,
        'metadata': {'description': f"Analysis (e) - {config['desc']}"}
    }
    with open(config['out'], 'wb') as f:
        pickle.dump(results_dict, f)
    print(f"✅ Shin Test p-value: {shin_res['p_value']:.4f} -> Saved to {config['out']}")

print("\n🎉 All operationalizations successfully executed and saved. Ready for final visualization.")